Obtain ID's of records that have zero subjects.  
Then free to experiment with methods to enrich their metadata.

In [182]:
NO_RECORDS = 1500

In [183]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
from tasks import transform_batch, add_file_metadata
import os
from fastapi import FastAPI, Query, HTTPException
import logging
from pydantic import BaseModel, Field
from utils.queue_utils import HarvestEventQueue, detect_identifier_type

from lxml import etree as ET
# API calls
import requests


# added for these experiments
from utils.enrichment_methods import get_doi, enrich_subjects_from_openaire

In [184]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [185]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")


if host == '':
    host = "127.0.0.1"
    user = os.getenv("POSTGRES_ADMIN")

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [186]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        array_to_string(
                xpath('oai:record/oai:metadata/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oaidc:oai_datacite/oaidc:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/},
                      {oaidc, http://schema.datacite.org/oai/oai-1.1/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oai:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
				-- note the 3 in datacite
                    '{{datacite, http://datacite.org/schema/kernel-3},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[] ||
				-- same path as previous but uses kernel-4 namespace
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[]
				, ', '
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    subjects
FROM query
WHERE query.subjects = ''
LIMIT %s
;
"""

In [187]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1721776245', 'title': 'Dynamic battery abuse testing with X-ray imaging', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/1721776245</identifier>\n    <datestamp>2026-04-29T08:26:59.286Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-1721776245</identifier><creators><creator><creatorName nameType="Per

In [188]:
record_titles

['Dynamic battery abuse testing with X-ray imaging',
 "Complexes of human butyrylcholinesterases with inhibitors as anti-Alzheimer's treatment",
 'Structural Biology at EMBL Grenoble, IBS-CIBB and IAB',
 'X-ray nano-tomography of mitochondria in H4 type cells',
 'Macromolecular Crystallography in Lyon/Toulouse/Bordeaux IBCP-MMSB / IPBS / IECB IBGC CBMN',
 'Dynamics of vesicles in dense pools studied by XPCS-microrheology (cont’d): extension to synaptic vesicles and active systems',
 'Charge Density Waves in YBa2Cu3O(7-d)',
 'Joint-Structrual Biology Group (JSBG) BAG proposal for ID29',
 'High-Energy Synchrotron Studies on Asteroid Samples Provided by Return Missions',
 'Austrian Crystallographic Diffraction Consortium',
 'Magneto structural phase transition in the frustrated metallic magnet Cr3Se4',
 'Structural Biology BAG - Portugal',
 'X-ray tomography and acoustic emissions of in situ, time-resolved magma fracturing: insights into the causes of explosive volcanic eruptions',
 'Expl

In [189]:
record_raw_metadata[2]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/1313792140</identifier>\n    <datestamp>2023-09-07T23:00:12.438Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-1313792140</identifier><creators><creator><creatorName nameType="Personal">Pauline JUYOUX</creatorName><givenName>Pauline</givenName><familyName>Juyoux</familyName><nameIdentifier nameIdentifierScheme="ORCID">

In [190]:
get_doi(record_raw_metadata[2])

'10.15151/ESRF-ES-1313792140'

# DOI check through bigger repositories  (#1)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Seaches thru OpenAlex or Openaire - apis?
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

OpenAIRE API
- they're officialy moving to a Graph API in May 2026
https://api.openaire.eu/graph/swagger-ui/index.html

In [191]:
import logging
logging.getLogger("urllib3").setLevel(logging.WARNING)

In [192]:
doi = get_doi(record_raw_metadata[0]) 

if not doi:
    print("NO DOI")
else:
    print(enrich_subjects_from_openaire(doi))

['PaN', 'Method and Instrumentation', 'x-ray probe', 'Particles, Nuclei and Fields', 'ID19', 'MI-1354']


In [193]:
# get only records that have a DOI
ix_has_doi = []

for i in range(0, len(record_raw_metadata)):
    rec = get_doi(record_raw_metadata[i])
    if rec:
        ix_has_doi.append(i)

print(ix_has_doi)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221,

In [194]:
id_to_enrich = [record_ids[i] for i in ix_has_doi]
doi_to_enrich = [get_doi(record_raw_metadata[i]) for i in ix_has_doi]

print(id_to_enrich)
print(doi_to_enrich)

['5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1721776245', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1313211663', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1313792140', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1838040359', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/96823860', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1690152590', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2313086718', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2239708400', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/799087423', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/624984240', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1931000089', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1471054035', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/584035539', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1833158337', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2399716666', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1119867040', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2290410954', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/223712195

In [195]:
# limit to smaller section
#id_to_enrich = id_to_enrich[:30]
#doi_to_enrich = doi_to_enrich[:30]

In [196]:
from utils.enrichment_methods import _openaire_batch_doi_resolve, _opeanaire_get_doi_subjects, _ids_connect_with_openaire_subjects, batch_enrich_subjects_from_openaire

api_result = _openaire_batch_doi_resolve(doi_to_enrich)

print(api_result[:3])

[{'authors': [{'fullName': 'VIOLOT, Sebastien', 'name': 'Sebastien', 'surname': 'VIOLOT', 'rank': 1, 'pid': None}, {'fullName': 'MOUREY, Lionel', 'name': 'Lionel', 'surname': 'MOUREY', 'rank': 2, 'pid': None}, {'fullName': 'TERRADOT-PIOT, Laurent', 'name': 'Laurent', 'surname': 'TERRADOT-PIOT', 'rank': 3, 'pid': None}, {'fullName': 'RAVAUD, Stephanie', 'name': 'Stephanie', 'surname': 'RAVAUD', 'rank': 4, 'pid': None}, {'fullName': 'GOUET, Patrice', 'name': 'Patrice', 'surname': 'GOUET', 'rank': 5, 'pid': None}, {'fullName': 'KAUFFMANN, Brice', 'name': 'Brice', 'surname': 'KAUFFMANN', 'rank': 6, 'pid': None}, {'fullName': 'GUILLET, Valerie', 'name': 'Valerie', 'surname': 'GUILLET', 'rank': 7, 'pid': None}, {'fullName': 'BALLUT, Lionel', 'name': 'Lionel', 'surname': 'BALLUT', 'rank': 8, 'pid': None}, {'fullName': 'CARIVENC, Coralie', 'name': 'Coralie', 'surname': 'CARIVENC', 'rank': 9, 'pid': None}, {'fullName': 'LONG, Mathieu', 'name': 'Mathieu', 'surname': 'LONG', 'rank': 10, 'pid': No

In [197]:
doi_subjects = _opeanaire_get_doi_subjects(api_result)
doi_subjects

[{'doi': '10.15151/esrf-es-96823860',
  'subjects': ['id23-2', 'Macromolecular Crystallography', 'MX-1979']},
 {'doi': '10.15151/esrf-es-1838040359',
  'subjects': ['x-ray probe', 'In-house Life Science', 'IH-LS-3636', 'ID16A']},
 {'doi': '10.15151/esrf-es-2313086718',
  'subjects': ['IH-HC-4243',
   'BM02',
   'In-house Hard Condensed Matter Science']},
 {'doi': '10.15151/esrf-es-2239708400',
  'subjects': ['MX-2731', 'Macromolecular Crystallography', 'ID29']},
 {'doi': '10.15151/esrf-es-1313792140',
  'subjects': ['PaN',
   'MX-2532',
   'Particles, Nuclei and Fields',
   'ID30A-1',
   'Macromolecular Crystallography']},
 {'doi': '10.15151/esrf-es-624984240',
  'subjects': ['PaN',
   'ID23-2',
   'MX-2354',
   'Particles, Nuclei and Fields',
   'Macromolecular Crystallography']},
 {'doi': '10.15151/esrf-es-1690152590',
  'subjects': ['PaN',
   'ID10',
   'x-ray probe',
   'Particles, Nuclei and Fields',
   'Soft Condensed Matter Science',
   'SC-5574']},
 {'doi': '10.15151/esrf-es-17

In [198]:
id_dois = []
for ix in range(len(id_to_enrich)):
    id_dois.append(
        {
            'id': id_to_enrich[ix],
            'doi': doi_to_enrich[ix].lower(),
        }
    )
id_dois

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1721776245',
  'doi': '10.15151/esrf-es-1721776245'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1313211663',
  'doi': '10.15151/esrf-es-1313211663'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1313792140',
  'doi': '10.15151/esrf-es-1313792140'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1838040359',
  'doi': '10.15151/esrf-es-1838040359'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/96823860',
  'doi': '10.15151/esrf-es-96823860'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1690152590',
  'doi': '10.15151/esrf-es-1690152590'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2313086718',
  'doi': '10.15151/esrf-es-2313086718'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2239708400',
  'doi': '10.15151/esrf-es-2239708400'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/799087423',
  'doi': '10.15151/esrf-es-799087423'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/624984240'

In [199]:
enriched_list = _ids_connect_with_openaire_subjects(id_dois, doi_subjects)
enriched_list

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1721776245',
  'doi': '10.15151/esrf-es-1721776245',
  'enriched_subjects': {'ID19',
   'MI-1354',
   'Method and Instrumentation',
   'PaN',
   'Particles, Nuclei and Fields',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1313211663',
  'doi': '10.15151/esrf-es-1313211663',
  'enriched_subjects': {'3. Good health',
   'A07-1-19',
   'BM07',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1313792140',
  'doi': '10.15151/esrf-es-1313792140',
  'enriched_subjects': {'ID30A-1',
   'MX-2532',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1838040359',
  'doi': '10.15151/esrf-es-1838040359',
  'enriched_subjects': {'ID16A',
   'IH-LS-3636',
   'In-house Life Science',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/96823860',
  'doi': '10.15151/esrf-es-96823

In [200]:
# Only succesfully enriched IDs and subjects.
# Could be useful for comparing how many ids we sent to be enriched vs
# how many actually got new subjects
[el for el in enriched_list if el['enriched_subjects'] is not None]

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1721776245',
  'doi': '10.15151/esrf-es-1721776245',
  'enriched_subjects': {'ID19',
   'MI-1354',
   'Method and Instrumentation',
   'PaN',
   'Particles, Nuclei and Fields',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1313211663',
  'doi': '10.15151/esrf-es-1313211663',
  'enriched_subjects': {'3. Good health',
   'A07-1-19',
   'BM07',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1313792140',
  'doi': '10.15151/esrf-es-1313792140',
  'enriched_subjects': {'ID30A-1',
   'MX-2532',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1838040359',
  'doi': '10.15151/esrf-es-1838040359',
  'enriched_subjects': {'ID16A',
   'IH-LS-3636',
   'In-house Life Science',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/96823860',
  'doi': '10.15151/esrf-es-96823

In [201]:
# All-in-one function

batch_enrich_subjects_from_openaire(id_dois)

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1721776245',
  'doi': '10.15151/esrf-es-1721776245',
  'enriched_subjects': {'ID19',
   'MI-1354',
   'Method and Instrumentation',
   'PaN',
   'Particles, Nuclei and Fields',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1313211663',
  'doi': '10.15151/esrf-es-1313211663',
  'enriched_subjects': {'3. Good health',
   'A07-1-19',
   'BM07',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1313792140',
  'doi': '10.15151/esrf-es-1313792140',
  'enriched_subjects': {'ID30A-1',
   'MX-2532',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1838040359',
  'doi': '10.15151/esrf-es-1838040359',
  'enriched_subjects': {'ID16A',
   'IH-LS-3636',
   'In-house Life Science',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/96823860',
  'doi': '10.15151/esrf-es-96823

In [202]:
# Comparison: Number of initial records vs number of records that have gained subjects via this method
enriched = batch_enrich_subjects_from_openaire(id_dois)
print(len([el['doi'] for el in enriched]))
print(len([el['doi'] for el in enriched if el['enriched_subjects'] is not None]))

1500
1499


In [203]:
# Taking a look at the metadata initially obtained from DB, subset of few records
full_info_enriched = [el for el in enriched if el['enriched_subjects'] is not None]

first_enriched = full_info_enriched[:10]

In [204]:
# indexes that have been enriched
ix_original = [record_ids.index(rec['id']) for rec in first_enriched]
ix_original

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [205]:
first_enriched

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1721776245',
  'doi': '10.15151/esrf-es-1721776245',
  'enriched_subjects': {'ID19',
   'MI-1354',
   'Method and Instrumentation',
   'PaN',
   'Particles, Nuclei and Fields',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1313211663',
  'doi': '10.15151/esrf-es-1313211663',
  'enriched_subjects': {'3. Good health',
   'A07-1-19',
   'BM07',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1313792140',
  'doi': '10.15151/esrf-es-1313792140',
  'enriched_subjects': {'ID30A-1',
   'MX-2532',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1838040359',
  'doi': '10.15151/esrf-es-1838040359',
  'enriched_subjects': {'ID16A',
   'IH-LS-3636',
   'In-house Life Science',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/96823860',
  'doi': '10.15151/esrf-es-96823

In [206]:
[record_raw_metadata[i] for i in ix_original]

['<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/1721776245</identifier>\n    <datestamp>2026-04-29T08:26:59.286Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-1721776245</identifier><creators><creator><creatorName nameType="Personal">Alexander Oliver RACK</creatorName><givenName>Alexander</givenName><familyName>Rack</familyName><nameIdentifier nameIdentifierScheme=

## OpenAlex API
https://developers.openalex.org/
https://developers.openalex.org/guides/authentication

Get multiple works by DOI

use | between DOIs, up to 100 DOIs at once!!
check out how the results look then.

https://api.openalex.org/works?filter=doi:https://doi.org/10.1234/a|https://doi.org/10.1234/b|https://doi.org/10.1234/c&per_page=100



Use the select parameter to return only specific fields:
https://api.openalex.org/works?select=id,doi,display_name

In [207]:
OpenAlex_API_key = os.getenv("OPENALEX_API_KEY")

In [208]:
OPENALEX_API_URL = "https://api.openalex.org/works/"

doi = '10.15151/esrf-es-1473312981'

# We have to append the ID directly in the path
query_url = OPENALEX_API_URL + "https://doi.org/" + doi

response = requests.get(
    query_url,
    params={"api_key": OpenAlex_API_key}, 
    timeout=30
)

data = response.json()


In [209]:
data

{'id': 'https://openalex.org/W6886229830',
 'doi': 'https://doi.org/10.15151/esrf-es-1473312981',
 'title': 'SAXS WAXS measurements on reference samples for different sample detector distances. Control of transmission measurements',
 'display_name': 'SAXS WAXS measurements on reference samples for different sample detector distances. Control of transmission measurements',
 'publication_year': 2024,
 'publication_date': '2024-01-25',
 'ids': {'openalex': 'https://openalex.org/W6886229830',
  'doi': 'https://doi.org/10.15151/esrf-es-1473312981'},
 'language': 'en',
 'primary_location': {'id': 'doi:10.15151/esrf-es-1473312981',
  'is_oa': True,
  'landing_page_url': 'https://doi.org/10.15151/esrf-es-1473312981',
  'pdf_url': None,
  'source': {'id': 'https://openalex.org/S7407051764',
   'display_name': 'European Synchrotron Radiation Facility',
   'issn_l': None,
   'issn': [],
   'is_oa': False,
   'is_in_doaj': False,
   'is_core': False,
   'host_organization': None,
   'host_organiza

In [210]:
oa_DOIS = [
    '10.15151/esrf-es-1001518064',
    '10.15151/esrf-es-2323561777',
    '10.15151/esrf-es-2323561778',
    '10.15151/esrf-es-2323561877',
    'https://doi.org/10.5286/ISIS.E.100051344',
    '10.34894/AUUSS2',
    '10.17026/DANS-28H-C53V',
    '10.34894/VLSN1K',
    '10.15151/ESRF-ES-1030397492',
    '10.17026/AR/DMHU53',
    '10.17026/AR/GHYQTO',
    '10.17026/DANS-ZU2-HF3N',
    '10.17026/DANS-ZTM-7SA5',
    '10.17026/DANS-ZUK-HN2Z',
    'https://doi.org/10.1016/j.paid.2021.110888',
    '10.23662/fors-ds-686-1'
]

In [211]:
OPENALEX_API_URL = "https://api.openalex.org/works"

def _openalex_batch_doi_resolve(dois, api_key):
    all_results = []
    chunk_size = 100

    # Strip any existing https://doi.org/ or http://doi.org/ prefix
    # so we can add it back consistently for every DOI
    clean_dois = [
        doi.replace("https://doi.org/", "")
           .replace("http://doi.org/", "")
        for doi in dois
    ]

    for i in range(0, len(clean_dois), chunk_size):
        chunk = clean_dois[i : i + chunk_size]

        # Join with pipe — each DOI gets the full https://doi.org/ prefix
        doi_filter = "|".join(f"https://doi.org/{doi}" for doi in chunk)
        # Build URL as f-string to avoid requests encoding the colons and pipes
        url = f"{OPENALEX_API_URL}?filter=doi:{doi_filter}&per_page={chunk_size}&api_key={api_key}"

        response = requests.get(url, timeout=30)
        response.raise_for_status()
        all_results.extend(response.json().get("results", []))

    return all_results

In [212]:
results = _openalex_batch_doi_resolve(oa_DOIS,
                    api_key=OpenAlex_API_key)
print(len(results))
results

2


[{'id': 'https://openalex.org/W3152931024',
  'doi': 'https://doi.org/10.1016/j.paid.2021.110888',
  'title': 'Personality, anxiety, and cognitive failures as predictors of flow proneness',
  'display_name': 'Personality, anxiety, and cognitive failures as predictors of flow proneness',
  'publication_year': 2021,
  'publication_date': '2021-04-08',
  'ids': {'openalex': 'https://openalex.org/W3152931024',
   'doi': 'https://doi.org/10.1016/j.paid.2021.110888',
   'mag': '3152931024'},
  'language': 'en',
  'primary_location': {'id': 'doi:10.1016/j.paid.2021.110888',
   'is_oa': False,
   'landing_page_url': 'https://doi.org/10.1016/j.paid.2021.110888',
   'pdf_url': None,
   'source': {'id': 'https://openalex.org/S151331055',
    'display_name': 'Personality and Individual Differences',
    'issn_l': '0191-8869',
    'issn': ['0191-8869', '1873-3549'],
    'is_oa': False,
    'is_in_doaj': False,
    'is_core': True,
    'host_organization': 'https://openalex.org/P4310320990',
    'ho

In [213]:
# Get potential subjects, two relevant fields: keywords and concepts
# Concepts is an older system, keywords is newer - but concepts is more diverse

In [214]:
kw_lists = [rez['keywords'] for rez in results]

keywords = [[item['display_name'] for item in sublist] for sublist in kw_lists]

keywords

[['Psychology',
  'Conscientiousness',
  'Extraversion and introversion',
  'Neuroticism',
  'Openness to experience',
  'Context (archaeology)',
  'Anxiety',
  'Personality',
  'Big Five personality traits',
  'Developmental psychology',
  'Clinical psychology',
  'Social psychology',
  'Psychiatry'],
 ['Cell',
  'Extracellular matrix',
  'Pathogenesis',
  'Protein folding',
  'Function (biology)',
  'Enzyme',
  'Cell adhesion',
  'Folding (DSP implementation)',
  'Gene']]

In [215]:
# Introduce a threshold of score:
keywords = [[item['display_name'] for item in sublist if item['score'] > 0.6] for sublist in kw_lists]

keywords

[['Psychology',
  'Conscientiousness',
  'Extraversion and introversion',
  'Neuroticism',
  'Openness to experience'],
 []]

In [216]:
def _opeanalex_get_doi_subjects(response: list[dict], score_treshold: float = 0.5):
    """
    Transform API response into a list of DOI-sets and their subject-sets.

    A single OpenAIRE ID will result in one or more DOIs and zero or more subjects.
    If is has no linked subjects, we can't enrich anything so we skip that ID entirely.
    The OpenAIRE ID itself doesn't concern us so it is not retained. 
    
    :param response: List[dict], the API response
    :return: List[dict]
        dict:
            DOI - DOI that is linked to at least one subject
            [Subjects] - all subjects for the given DOI
    """
    if response == []:
        # no DOIs with any results
        return []
    
    # sets of DOIs with relevant sets of subjects. Later, spread to individual DOIs
    enriched_dois = []
    enriched_subjects = []

    for el in response:
        kw_list = el['keywords']

        if (kw_list is None or kw_list == []):
            # this element doesn't contribute to enriching subjects,
            # so even the DOIs don't matter
            continue

        keywords = [item['display_name'] for item in kw_list if item['score'] > score_treshold]

        # After filtering by score it's possible that no keywords are left,
        # Then this DOI won't be enriched
        if not keywords:
            continue
        
        current_doi = el['doi']

        # Strip it from prefixes and make it lowercase
        current_doi = current_doi.replace("https://doi.org/", "").replace("http://doi.org/", "").lower()

        enriched_dois.append(current_doi)

        enriched_subjects.append(keywords)

    # Create a list of dicts as output

    doi_subjects = []

    for ix in range(len(enriched_dois)):
        doi_sub_pair = {'doi': enriched_dois[ix],
                        'subjects': enriched_subjects[ix]}
        doi_subjects.append(doi_sub_pair)

    return doi_subjects

In [217]:
_opeanalex_get_doi_subjects(results)

[{'doi': '10.1016/j.paid.2021.110888',
  'subjects': ['Psychology',
   'Conscientiousness',
   'Extraversion and introversion',
   'Neuroticism',
   'Openness to experience',
   'Context (archaeology)',
   'Anxiety',
   'Personality',
   'Big Five personality traits']},
 {'doi': '10.15151/esrf-es-2323561777',
  'subjects': ['Cell',
   'Extracellular matrix',
   'Pathogenesis',
   'Protein folding',
   'Function (biology)']}]

In [218]:
_opeanalex_get_doi_subjects(results, score_treshold=0.65)

[{'doi': '10.1016/j.paid.2021.110888',
  'subjects': ['Psychology',
   'Conscientiousness',
   'Extraversion and introversion',
   'Neuroticism',
   'Openness to experience']}]

In [219]:
_opeanalex_get_doi_subjects(results, score_treshold=0)

[{'doi': '10.1016/j.paid.2021.110888',
  'subjects': ['Psychology',
   'Conscientiousness',
   'Extraversion and introversion',
   'Neuroticism',
   'Openness to experience',
   'Context (archaeology)',
   'Anxiety',
   'Personality',
   'Big Five personality traits',
   'Developmental psychology',
   'Clinical psychology',
   'Social psychology',
   'Psychiatry']},
 {'doi': '10.15151/esrf-es-2323561777',
  'subjects': ['Cell',
   'Extracellular matrix',
   'Pathogenesis',
   'Protein folding',
   'Function (biology)',
   'Enzyme',
   'Cell adhesion',
   'Folding (DSP implementation)',
   'Gene']}]

In [220]:
id_dois

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1721776245',
  'doi': '10.15151/esrf-es-1721776245'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1313211663',
  'doi': '10.15151/esrf-es-1313211663'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1313792140',
  'doi': '10.15151/esrf-es-1313792140'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1838040359',
  'doi': '10.15151/esrf-es-1838040359'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/96823860',
  'doi': '10.15151/esrf-es-96823860'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1690152590',
  'doi': '10.15151/esrf-es-1690152590'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2313086718',
  'doi': '10.15151/esrf-es-2313086718'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2239708400',
  'doi': '10.15151/esrf-es-2239708400'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/799087423',
  'doi': '10.15151/esrf-es-799087423'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/624984240'

In [221]:
res = _openalex_batch_doi_resolve([el['doi'] for el in id_dois],
                            OpenAlex_API_key)

subj = _opeanalex_get_doi_subjects(res, score_treshold=0)

subj

[{'doi': '10.15151/esrf-es-2237121954',
  'subjects': ['Dehydrogenation',
   'Oxidative phosphorylation',
   'Catalysis',
   'Redox',
   'Molybdate',
   'Raman spectroscopy',
   'In situ',
   'Ethanol']},
 {'doi': '10.15151/esrf-es-2240732946',
  'subjects': ['Electronics',
   'Microscopy',
   'Power electronics',
   'Field (mathematics)',
   'Software',
   'Digitization',
   'Power (physics)',
   'Semiconductor device',
   'Semiconductor']},
 {'doi': '10.15151/esrf-es-2283072140',
  'subjects': ['Polarimetry',
   'Backscatter (email)',
   'Scattering',
   'Reflection (computer programming)',
   'Field (mathematics)']},
 {'doi': '10.15151/esrf-es-2290410954',
  'subjects': ['Ultrashort pulse',
   'Monoclinic crystal system',
   'Phase transition',
   'Phase (matter)',
   'Anisotropy',
   'Diffraction',
   'Single crystal']},
 {'doi': '10.15151/esrf-es-2313086718',
  'subjects': ['Charge density',
   'Charge (physics)',
   'Charge density wave',
   'Copper',
   'Work (physics)']},
 {'do

In [222]:
len(res)

194

In [223]:
connected = _ids_connect_with_openaire_subjects(id_dois, subj)
connected

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1721776245',
  'doi': '10.15151/esrf-es-1721776245',
  'enriched_subjects': None},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1313211663',
  'doi': '10.15151/esrf-es-1313211663',
  'enriched_subjects': None},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1313792140',
  'doi': '10.15151/esrf-es-1313792140',
  'enriched_subjects': None},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1838040359',
  'doi': '10.15151/esrf-es-1838040359',
  'enriched_subjects': None},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/96823860',
  'doi': '10.15151/esrf-es-96823860',
  'enriched_subjects': None},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1690152590',
  'doi': '10.15151/esrf-es-1690152590',
  'enriched_subjects': None},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2313086718',
  'doi': '10.15151/esrf-es-2313086718',
  'enriched_subjects': {'Charge (physics)',
   'Charge density',
   'Charge density wave',
   'C

In [224]:
successfully_enriched = [el for el in connected if el['enriched_subjects'] is not None]
print(len(successfully_enriched))
successfully_enriched

182


[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2313086718',
  'doi': '10.15151/esrf-es-2313086718',
  'enriched_subjects': {'Charge (physics)',
   'Charge density',
   'Charge density wave',
   'Copper',
   'Work (physics)'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2399716666',
  'doi': '10.15151/esrf-es-2399716666',
  'enriched_subjects': {'Automotive battery',
   'Battery (electricity)',
   'Flexibility (engineering)',
   'Power (physics)',
   'Work (physics)'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2290410954',
  'doi': '10.15151/esrf-es-2290410954',
  'enriched_subjects': {'Anisotropy',
   'Diffraction',
   'Monoclinic crystal system',
   'Phase (matter)',
   'Phase transition',
   'Single crystal',
   'Ultrashort pulse'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2237121954',
  'doi': '10.15151/esrf-es-2237121954',
  'enriched_subjects': {'Catalysis',
   'Dehydrogenation',
   'Ethanol',
   'In situ',
   'Molybdate',
   'Oxidative phosphoryla

In [225]:
len(id_dois)

1500

In [226]:
enr_doi = [el['doi'] for el in successfully_enriched]

res_doi = [el['doi'] for el in res]

subj_doi = [el['doi'] for el in subj]

In [227]:
print(len(enr_doi))
print(len(res_doi))

182
194


In [228]:
[el for el in res if el['doi'].replace("https://doi.org/", "") not in enr_doi]

[{'id': 'https://openalex.org/W7116073578',
  'doi': 'https://doi.org/10.15151/esrf-es-2303014306',
  'title': 'XMCD on REIr2Si2 intermetallics',
  'display_name': 'XMCD on REIr2Si2 intermetallics',
  'publication_year': 2025,
  'publication_date': '2025-12-02',
  'ids': {'openalex': 'https://openalex.org/W7116073578',
   'doi': 'https://doi.org/10.15151/esrf-es-2303014306'},
  'language': 'en',
  'primary_location': {'id': 'doi:10.15151/esrf-es-2303014306',
   'is_oa': True,
   'landing_page_url': 'https://doi.org/10.15151/esrf-es-2303014306',
   'pdf_url': None,
   'source': {'id': 'https://openalex.org/S7407051764',
    'display_name': 'European Synchrotron Radiation Facility',
    'issn_l': None,
    'issn': [],
    'is_oa': False,
    'is_in_doaj': False,
    'is_core': False,
    'host_organization': None,
    'host_organization_name': None,
    'host_organization_lineage': [],
    'host_organization_lineage_names': [],
    'type': 'repository'},
   'license': 'cc-by',
   'licens

In [229]:
input_dois = {el['doi'] for el in id_dois}
result_dois = {el['doi'] for el in subj}

print(len(result_dois - input_dois))
result_dois - input_dois

0


set()

In [230]:
print(len(input_dois - result_dois))
input_dois - result_dois

1318


{'10.15151/esrf-es-1356709575',
 '10.15151/esrf-es-2250350908',
 '10.15151/esrf-es-1313930306',
 '10.15151/esrf-es-1307922306',
 '10.15151/esrf-es-778016223',
 '10.15151/esrf-es-2137660175',
 '10.15151/esrf-es-686522079',
 '10.15151/esrf-es-2005116482',
 '10.15151/esrf-es-1168800372',
 '10.15151/esrf-es-1706795788',
 '10.15151/esrf-es-1548586886',
 '10.15151/esrf-es-771421677',
 '10.15151/esrf-es-1651573809',
 '10.15151/esrf-es-986776249',
 '10.15151/esrf-es-698653440',
 '10.15151/esrf-es-925343097',
 '10.15151/esrf-es-1838749251',
 '10.15151/esrf-es-1566400240',
 '10.15151/esrf-es-2205320498',
 '10.15151/esrf-es-90632248',
 '10.15151/esrf-es-2134382780',
 '10.15151/esrf-es-186965553',
 '10.15151/esrf-es-1296308413',
 '10.15151/esrf-es-1504462336',
 '10.15151/esrf-es-1830136633',
 '10.15151/esrf-es-1018421213',
 '10.15151/esrf-es-2114505374',
 '10.15151/esrf-es-1174997139',
 '10.15151/esrf-es-645854790',
 '10.15151/esrf-es-765444340',
 '10.15151/esrf-es-424006209',
 '10.15151/esrf-es-1

In [232]:
api_result_dois = {el['doi'].replace("https://doi.org/", "").lower() for el in res}
api_result_dois

{'10.15151/esrf-es-1031684904',
 '10.15151/esrf-es-2223371281',
 '10.15151/esrf-es-2231628049',
 '10.15151/esrf-es-2232201338',
 '10.15151/esrf-es-2232203725',
 '10.15151/esrf-es-2232211521',
 '10.15151/esrf-es-2233033828',
 '10.15151/esrf-es-2233628787',
 '10.15151/esrf-es-2233746730',
 '10.15151/esrf-es-2233749970',
 '10.15151/esrf-es-2233750161',
 '10.15151/esrf-es-2233785992',
 '10.15151/esrf-es-2233786162',
 '10.15151/esrf-es-2233786756',
 '10.15151/esrf-es-2233788342',
 '10.15151/esrf-es-2233798033',
 '10.15151/esrf-es-2233798281',
 '10.15151/esrf-es-2234399915',
 '10.15151/esrf-es-2234936903',
 '10.15151/esrf-es-2237121954',
 '10.15151/esrf-es-2238049433',
 '10.15151/esrf-es-2238176699',
 '10.15151/esrf-es-2238913770',
 '10.15151/esrf-es-2240732946',
 '10.15151/esrf-es-2241561320',
 '10.15151/esrf-es-2241562517',
 '10.15151/esrf-es-2241573073',
 '10.15151/esrf-es-2241573092',
 '10.15151/esrf-es-2243235796',
 '10.15151/esrf-es-2248514742',
 '10.15151/esrf-es-2249474906',
 '10.151

In [235]:
api_result_dois - input_dois

set()

In [236]:
res_not_enriching = api_result_dois - result_dois
res_not_enriching

{'10.15151/esrf-es-2274266070',
 '10.15151/esrf-es-2276136404',
 '10.15151/esrf-es-2276581460',
 '10.15151/esrf-es-2303014306',
 '10.15151/esrf-es-2305776132',
 '10.15151/esrf-es-2310096266',
 '10.15151/esrf-es-2359114843',
 '10.15151/esrf-es-2362526947',
 '10.15151/esrf-es-2363023602',
 '10.15151/esrf-es-2378068614',
 '10.15151/esrf-es-2390593336',
 '10.15151/esrf-es-2400148510'}

In [237]:
res_ne_info = [el for el in res if el['doi'].replace("https://doi.org/", "").lower() in res_not_enriching]
len(res_ne_info)

12

In [241]:
[el['keywords'] for el in res_ne_info]

[[], [], [], [], [], [], [], [], [], [], [], []]

the API result object can have a result for a given DOI, but that result doesn't contain any keywords = doesn't contribute to enrichment
Hence the len(api_result) can be larger than len(enriched_dois)

Additionaly if we set a high keyword score threshold, a certain result MIGHT have keywords but none are relevant enough - that will also not contribute to enrichment

# DOI resolver method  (#2)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Talks to an API such as DataCite or SignPosting.
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword